# Build Logic Model Dataset
Author: Alemarie Ceria <br>
Last Updated: 03/5/26

## Setup

### Imports

In [63]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import geopandas as gpd
import pandas as pd
import numpy as np

### Configurations

In [64]:
# Today
TODAY = datetime.now(ZoneInfo("Pacific/Honolulu")).strftime("%Y%m%d")

### Paths

In [65]:
# Project root (assumes notebook is in `project/notebooks/`)
project_root = Path.cwd().parent

# Directories
data_dir = project_root / "data"
processed_dir  = data_dir / "03_processed"
mpat_dir = processed_dir / "mpat"
logic_dir = processed_dir / "logic"

# File paths
mpat_v_date = "20260301"
file_paths = {
    # MPAT
    "input": {"path": mpat_dir / f"{mpat_v_date}_mpat_32604.gpkg", "layer": "mpat"},
    "output_gpkg": logic_dir / f"{TODAY}_logic_32604.gpkg",
    "output_csv": logic_dir / f"{TODAY}_logic.csv",
}

## Load Master Parcel Attributes Table (MPAT)

In [66]:
mpat_gdf = gpd.read_file(file_paths["input"]["path"], layer=file_paths["input"]["layer"])
mpat_gdf.head()

,island,tmk,osds_qty,bedroom_qty,building_fp_qty,parcel_area_sqft,building_fp_total_area_sqft,net_parcel_area_sqft,dist_to_sma_ft,dist_to_coast_ft,...,ksat_l,ksat_r,avg_rainfall_in,land_surface_elev_ft,wt_elev_ft,depth_to_wt_ft,slope_pct,sfha_tf,analysis_point_source,geometry
0,Maui,211003003,1,3,2.0,70924.721493,3445.443843,67479.277650,0.0,204.446376,...,6.546667,13.333333,106.080597,7.293545,0.328084,6.965461,2.190298,T,building_fp_largest_centroid,"MULTIPOLYGON (((797198.639 2309554.974, 797196..."
1,Maui,211003006,1,2,2.0,32346.536954,2271.544397,30074.992557,0.0,192.521771,...,6.546667,13.333333,110.287704,7.958200,0.328084,7.630116,2.336785,T,building_fp_largest_centroid,"MULTIPOLYGON (((797225.075 2309335.433, 797159..."
2,Maui,211003012,1,2,2.0,38854.544707,1777.038765,37077.505942,0.0,274.249920,...,6.546667,13.333333,110.287704,13.798596,0.328084,13.470512,3.301470,T,building_fp_largest_centroid,"MULTIPOLYGON (((797153.617 2309339.286, 797107..."
3,Maui,211003030,1,3,1.0,12086.822566,661.600642,11425.221924,0.0,1111.869936,...,14.000000,28.000000,121.329597,44.486740,0.328084,44.158656,18.649988,F,building_fp_largest_centroid,"MULTIPOLYGON (((796828.023 2309063.058, 796822..."
4,Maui,211003031,1,1,NaN,129395.545175,NaN,NaN,0.0,868.201418,...,14.000000,28.000000,121.329597,42.756844,0.328084,42.428760,22.219120,F,parcel_centroid,"MULTIPOLYGON (((796820.317 2309089.443, 796830..."


## Variable Classifications and Gate Flags

### Depth to Water Table

In [67]:
depth_to_wt_suitability = (
    mpat_gdf[["tmk", "depth_to_wt_ft"]]
    .assign(
        class_depth_to_wt=lambda d: pd.Categorical(
            np.select(
                [
                    d["depth_to_wt_ft"] < 3,
                    d["depth_to_wt_ft"].between(3, 6, inclusive="both"),
                    d["depth_to_wt_ft"] > 6,
                ],
                [
                    "Less than 3 ft",
                    "Between 3 and 6 ft",
                    "Greater than 6 ft",
                ],
                default=pd.NA,
            ),
            categories=[
                "Less than 3 ft",
                "Between 3 and 6 ft",
                "Greater than 6 ft",
            ],
            ordered=True,
        ),
        flag_depth_to_wt=lambda d: d["depth_to_wt_ft"].lt(3).where(
            d["depth_to_wt_ft"].notna() & d["depth_to_wt_ft"].le(500)
        ).astype("Int64"),
    )
)
print(f"{len(depth_to_wt_suitability)}\n")
print(f"Missing class_depth_to_wt: {depth_to_wt_suitability['class_depth_to_wt'].isna().sum()}")
print(f"Missing flag_depth_to_wt: {depth_to_wt_suitability['flag_depth_to_wt'].isna().sum()} ({depth_to_wt_suitability['flag_depth_to_wt'].isna().sum() / len(depth_to_wt_suitability) * 100:.0f}%)")
print(f"WTD > 500 ft: {(mpat_gdf['depth_to_wt_ft'] > 500).sum()} ({(mpat_gdf['depth_to_wt_ft'] > 500).sum() / len(mpat_gdf) * 100:.0f}%)\n")
depth_to_wt_suitability.head()

7505

Missing class_depth_to_wt: 1
Missing flag_depth_to_wt: 5404 (72%)
WTD > 500 ft: 5403 (72%)



,tmk,depth_to_wt_ft,class_depth_to_wt,flag_depth_to_wt
0,211003003,6.965461,Greater than 6 ft,0
1,211003006,7.630116,Greater than 6 ft,0
2,211003012,13.470512,Greater than 6 ft,0
3,211003030,44.158656,Greater than 6 ft,0
4,211003031,42.428760,Greater than 6 ft,0


**Data quality notes:**
- `flag_depth_to_wt` is null for 5,404 parcels (72%)
- 5403 cases where depth_to_wt_ft > 500 ft after subtracting from land surface elevation
    - 482 parcels had true NoData from the water table raster and were filled with 0.1 m or 0.328084 ft (sea level) per Chris's instruction
    - ~4,921 additional parcels where the raster returned near sea level values at higher elevation locations.
- One additional parcel has a null land surface elevation (outside DEM extent). 

**Questions for Chris/Robert:**
- Does the water table raster have valid coverage for high-elevation parcels, or is the near sea level return a known raster limitation?
- Should parcels with `depth_to_wt_ft` > 500 ft be excluded from the WTD gate entirely, or assigned a default pass (standard septic tank eligible)?
- Should the 482 NoData parcels (filled with sea level) be handled differently from the 5,403 parcels with implausible raster returns?

### Lot Size

In [68]:
lot_size_req = (
    mpat_gdf[["tmk", "net_parcel_area_sqft"]]
    .assign(
        class_lot_size=lambda d: pd.Categorical(
            np.select(
                [
                    d["net_parcel_area_sqft"] < 10_000,
                    d["net_parcel_area_sqft"].between(10_000, 21_000, inclusive="both"),
                    d["net_parcel_area_sqft"] > 21_000,
                ],
                [
                    "Less than 10,000 sqft",
                    "Between 10,000 and 21,000 sqft",
                    "Greater than 21,000 sqft",
                ],
                default=pd.NA,
            ),
            categories=[
                "Less than 10,000 sqft",
                "Between 10,000 and 21,000 sqft",
                "Greater than 21,000 sqft",
            ],
            ordered=True,
        ),
        flag_lot_size=lambda d: d["net_parcel_area_sqft"].lt(10_000).where(
            d["net_parcel_area_sqft"].notna()
        ).astype("Int64"),
    )
)
print(f"{len(lot_size_req)}\n")
print(f"{lot_size_req["flag_lot_size"].value_counts()}\n")
print(f"Missing class_lot_size: {lot_size_req['class_lot_size'].isna().sum()}")
print(f"Missing flag_lot_size: {lot_size_req['flag_lot_size'].isna().sum()}")
print(f"Negative net_parcel_area_sqft: {(mpat_gdf['net_parcel_area_sqft'] < 0).sum()}\n")
lot_size_req.head()

7505

flag_lot_size
0    5171
1    2028
Name: count, dtype: Int64

Missing class_lot_size: 306
Missing flag_lot_size: 306
Negative net_parcel_area_sqft: 8



,tmk,net_parcel_area_sqft,class_lot_size,flag_lot_size
0,211003003,67479.277650,"Greater than 21,000 sqft",0
1,211003006,30074.992557,"Greater than 21,000 sqft",0
2,211003012,37077.505942,"Greater than 21,000 sqft",0
3,211003030,11425.221924,"Between 10,000 and 21,000 sqft",0
4,211003031,NaN,NaN,<NA>


### Slope

In [69]:
slope_req = (
    mpat_gdf[["tmk", "slope_pct"]]
    .assign(
        class_slope=lambda d: pd.Categorical(
            np.select(
                [
                    d["slope_pct"] < 8,
                    d["slope_pct"].between(8, 12, inclusive="both"),
                    d["slope_pct"] > 12,
                ],
                [
                    "Less than 8%",
                    "Between 8 and 12%",
                    "Greater than 12%",
                ],
                default=pd.NA,
            ),
            categories=[
                "Less than 8%",
                "Between 8 and 12%",
                "Greater than 12%",
            ],
            ordered=True,
        ),
        flag_slope=lambda d: d["slope_pct"].gt(12.0).where(
            d["slope_pct"].notna()
        ).astype("Int64"),
    )
)

print(f"{len(slope_req)}\n")
print(f"{slope_req['flag_slope'].value_counts()}\n")
print(f"Missing class_slope: {slope_req['class_slope'].isna().sum()}")
print(f"Missing flag_slope:  {slope_req['flag_slope'].isna().sum()}")
slope_req.head()

7505

flag_slope
0    5033
1    2471
Name: count, dtype: Int64

Missing class_slope: 1
Missing flag_slope:  1


,tmk,slope_pct,class_slope,flag_slope
0,211003003,2.190298,Less than 8%,0
1,211003006,2.336785,Less than 8%,0
2,211003012,3.301470,Less than 8%,0
3,211003030,18.649988,Greater than 12%,1
4,211003031,22.219120,Greater than 12%,1


## Aggregation and Technology Recommendation

In [70]:
logic_gdf = (
    mpat_gdf[["tmk", "geometry"]]
    # Join all three variable tables
    .merge(depth_to_wt_suitability[["tmk", "class_depth_to_wt", "flag_depth_to_wt"]], on="tmk", how="left", validate="one_to_one")
    .merge(lot_size_req[["tmk", "class_lot_size", "flag_lot_size"]], on="tmk", how="left", validate="one_to_one")
    .merge(slope_req[["tmk", "class_slope", "flag_slope"]], on="tmk", how="left", validate="one_to_one")
    # Aggregation, sum of flags (NA flags excluded from sum)
    .assign(
        flag_count=lambda d: (
            d[["flag_depth_to_wt", "flag_lot_size", "flag_slope"]]
            # Parcel with one flag and two NAs will still get a count
            # Only treated as NA if all three flags are NA
            .sum(axis=1, skipna=True)
            .astype("Int64")
        ),
        # Technology recommendation
        recommendation=lambda d: pd.Categorical(
            np.select(
                [
                    d["flag_count"] >= 1,
                    d["flag_count"] == 0,
                ],
                [
                    "ATU NSF 40",
                    "Standard Septic Tank",
                ],
                default=pd.NA,
            ),
            categories=["Standard Septic Tank", "ATU NSF 40"],
            ordered=True,
        ),
    )
    # Relocate geometry to end
    .pipe(lambda d: d[[c for c in d.columns if c != "geometry"] + ["geometry"]])
)

print(f"{len(logic_gdf)}\n")
print(logic_gdf["recommendation"].value_counts(dropna=False))
print(f"\nMissing recommendation: {logic_gdf['recommendation'].isna().sum()}")
print(f"Missing flag_count: {logic_gdf['flag_count'].isna().sum()}")
logic_gdf.head()

7505

recommendation
ATU NSF 40              4204
Standard Septic Tank    3301
Name: count, dtype: int64

Missing recommendation: 0
Missing flag_count: 0


,tmk,class_depth_to_wt,flag_depth_to_wt,class_lot_size,flag_lot_size,class_slope,flag_slope,flag_count,recommendation,geometry
0,211003003,Greater than 6 ft,0,"Greater than 21,000 sqft",0,Less than 8%,0,0,Standard Septic Tank,"MULTIPOLYGON (((797198.639 2309554.974, 797196..."
1,211003006,Greater than 6 ft,0,"Greater than 21,000 sqft",0,Less than 8%,0,0,Standard Septic Tank,"MULTIPOLYGON (((797225.075 2309335.433, 797159..."
2,211003012,Greater than 6 ft,0,"Greater than 21,000 sqft",0,Less than 8%,0,0,Standard Septic Tank,"MULTIPOLYGON (((797153.617 2309339.286, 797107..."
3,211003030,Greater than 6 ft,0,"Between 10,000 and 21,000 sqft",0,Greater than 12%,1,1,ATU NSF 40,"MULTIPOLYGON (((796828.023 2309063.058, 796822..."
4,211003031,Greater than 6 ft,0,NaN,<NA>,Greater than 12%,1,1,ATU NSF 40,"MULTIPOLYGON (((796820.317 2309089.443, 796830..."


## Export

**Note:** Flag columns export as float64 in CSV due to NA serialization. When reading in, convert back to integer.

In [ ]:
# Ensure output dir exists
logic_dir.mkdir(parents=True, exist_ok=True)

# Export logic data as GeoPackage
logic_layer = "logic"
logic_gdf.to_file(file_paths["output_gpkg"], layer=logic_layer, driver="GPKG")
print("Wrote GPKG:", file_paths["output_gpkg"])

# Export logic data as CSV (drops geometry)
logic_csv_df = logic_gdf.drop(columns=["geometry"], errors="ignore")
logic_csv_df.to_csv(file_paths["output_csv"], index=False)
print("Wrote CSV:", file_paths["output_csv"])

## Decision Flowchart Check

In [72]:
sample = logic_gdf.head(5).copy()

print("=" * 60)
print("Decision flowchart check (first 5 parcels)")
print("=" * 60)

for _, row in sample.iterrows():
    print(f"\nTMK: {row['tmk']}")
    print(f"  Gate 1 — Depth to water table < 3 ft?")
    print(f"    depth_to_wt class : {row['class_depth_to_wt']}")
    print(f"    flag_depth_to_wt  : {row['flag_depth_to_wt']}")
    
    print(f"  Gate 2 — Lot size < 10,000 sqft?")
    print(f"    class_lot_size    : {row['class_lot_size']}")
    print(f"    flag_lot_size     : {row['flag_lot_size']}")
    
    print(f"  Gate 3 — Slope > 12%?")
    print(f"    class_slope       : {row['class_slope']}")
    print(f"    flag_slope        : {row['flag_slope']}")
    
    print(f"  Aggregation — any ATU flags raised?")
    print(f"    flag_count        : {row['flag_count']}")
    print(f"    → Recommendation  : {row['recommendation']}")

Decision flowchart check (first 5 parcels)

TMK: 211003003
  Gate 1 — Depth to water table < 3 ft?
    depth_to_wt class : Greater than 6 ft
    flag_depth_to_wt  : 0
  Gate 2 — Lot size < 10,000 sqft?
    class_lot_size    : Greater than 21,000 sqft
    flag_lot_size     : 0
  Gate 3 — Slope > 12%?
    class_slope       : Less than 8%
    flag_slope        : 0
  Aggregation — any ATU flags raised?
    flag_count        : 0
    → Recommendation  : Standard Septic Tank

TMK: 211003006
  Gate 1 — Depth to water table < 3 ft?
    depth_to_wt class : Greater than 6 ft
    flag_depth_to_wt  : 0
  Gate 2 — Lot size < 10,000 sqft?
    class_lot_size    : Greater than 21,000 sqft
    flag_lot_size     : 0
  Gate 3 — Slope > 12%?
    class_slope       : Less than 8%
    flag_slope        : 0
  Aggregation — any ATU flags raised?
    flag_count        : 0
    → Recommendation  : Standard Septic Tank

TMK: 211003012
  Gate 1 — Depth to water table < 3 ft?
    depth_to_wt class : Greater than 6 ft

## Deferred Variables

The following variables are excluded from this version pending source document verification and feedback.

- SMA constraints (`dist_to_sma_ft`)
- Climate suitability (`avg_rainfall_in`)
- Flood zone (`sfha_tf`)
- Coastline proximity (`dist_to_coast_ft`)
- Stream proximity (`dist_to_streams_ft`)

In [73]:
### SMA Constraints

# sma_constraints = (
#     mpat_gdf[["tmk", "dist_to_sma_ft"]]
#     .assign(
#         sma_constraints=lambda d: pd.Categorical(
#             np.select(
#                 [
#                     d["dist_to_sma_ft"] <= 50,
#                     d["dist_to_sma_ft"] > 50,
#                 ],
#                 ["Within 50 ft", "Beyond 50 ft"],
#                 default=pd.NA,
#             ),
#             categories=["Within 50 ft", "Beyond 50 ft"],
#             ordered=True,
#         )
#     )
# )
# print(len(sma_constraints))
# sma_constraints.head()

# Values of 0.0 mean the analysis point falls within an SMA. 

In [74]:
### Climate Suitability

# climate_suitability = (
#     mpat_gdf[["tmk", "avg_rainfall_in"]]
#     .assign(
#         climate_suitability=lambda d: pd.Categorical(
#             np.select(
#                 [
#                     d["avg_rainfall_in"] <= 20,
#                     (d["avg_rainfall_in"] > 20) & (d["avg_rainfall_in"] < 75),
#                     d["avg_rainfall_in"] >= 75,
#                 ],
#                 ["Xeric", "Mesic", "Hydric"],
#                 default=pd.NA,
#             ),
#             categories=["Xeric", "Mesic", "Hydric"],
#             ordered=True,
#         )
#     )
# )
# print(len(climate_suitability))
# climate_suitability.head()

In [75]:
### Within Flood Zone

# within_flood_zone_gdf = (
#     mpat_gdf[["tmk", "sfha_tf"]]
#     .rename(columns={"sfha_tf": "in_flood_zone"})
#     .assign(in_flood_zone=lambda d: d["in_flood_zone"].map({"T": 1, "F": 0}).astype("int64"))
# )
# print(len(within_flood_zone_gdf))
# within_flood_zone_gdf.head()

In [76]:
### Within SMA

# Analysis point in SMA (distance == 0 ft)
# within_sma = mpat_gdf[["tmk", "dist_to_sma_ft"]].assign(in_sma=lambda d: (d["dist_to_sma_ft"] == 0.0).astype("int64"))
# print(len(within_sma))
# within_sma.head()

In [77]:
### Within 100 ft of Coastline

# coast_within_100_ft = mpat_gdf[["tmk", "dist_to_coast_ft"]].assign(coast_within_100_ft=lambda d: (d["dist_to_coast_ft"] <= 100).astype("int64"))
# print(len(coast_within_100_ft))
# coast_within_100_ft.head()

In [78]:
### Within 50 ft of a Stream

# stream_within_50_ft = mpat_gdf[["tmk", "dist_to_streams_ft"]].assign(stream_within_50_ft=lambda d: (d["dist_to_streams_ft"] <= 50).astype("int64"))
# print(len(stream_within_50_ft))
# stream_within_50_ft.head()